## C5_02 — Construirea vector store-ului pentru o bulă
În acest notebook construim un vector store FAISS pentru o singură bulă / un singur agent.
Fiecare student lucrează pe bula lui. Scopul este să vedem clar cum textele curățate devin embeddings, apoi index FAISS.
Mai târziu, aceeași logică va fi pusă într-un script `.py` care rulează automat pentru toate bulele.

## 0. Setup

In [4]:
pip install sentence_transformers

   ---------------------------------------- 0.0/588.7 kB ? eta -:--:--
   ---------------------------------------- 588.7/588.7 kB 7.2 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from pathlib import Path
import os, pickle
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

while not Path("data/bubbles").exists():
    os.chdir("..")

BUBBLES_DIR = Path("data/bubbles")
VECTOR_DIR = Path("assets/vectorstores")
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

c:\Users\User\Desktop\Analiza Datelor Complexe\AN2\curs_Inginerie_AI\proiect_AI\echochamber-project-team-4\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Aleg bula mea
Alege fișierul `.jsonl` al bulei tale.
Acest fișier a fost creat în etapa anterioară, după verificarea manuală a textelor.

In [6]:
MY_BUBBLE_FILE = "anti_sistem.jsonl" 

bubble_path = BUBBLES_DIR / MY_BUBBLE_FILE
slug = bubble_path.stem

df_bubble = pd.read_json(bubble_path, lines=True)

print("Bula:", slug)
print("Texte:", len(df_bubble))

df_bubble[["id", "agent", "text"]].head()

Bula: anti_sistem
Texte: 50


,id,agent,text
0,yt_33ypGncXCIw_UgxJ1WUJvAUiQsfh0kN4AaABAg,Anti-sistem,CU TIPETE DNA JUDECATOR NU SE LUCREAZA CU SETA...
1,yt_0b2_71jAaQs_UgyDrtDbtwBZRgFz4Jd4AaABAg,Anti-sistem,"D-le Președinte,dacă nu aveți curajul,onoarea,..."
2,yt_M37Lar0c11g_UgzqcSjMn379kcSJAZl4AaABAg,Anti-sistem,Da-ți programul vizitei lui Cerşinski în Român...
3,yt_bnbjgKIS5Dw_UgzlWxIqTksTiU4WQ2d4AaABAg,Anti-sistem,"SI INCA CEVA ,APROPO DE ,,POLITICA POPILOR: IN..."
4,yt_yEuctxNb4O0_UgxNv47481MET5yGVVh4AaABAg,Anti-sistem,"Domnule Președinte luați atitudine , cine știe..."


## 2. Pregătim textele
Pentru FAISS avem nevoie de o listă simplă de texte.
Metadata rămâne separat, ca să putem lega fiecare vector de textul original.

In [7]:
texts = df_bubble["text"].fillna("").tolist()
metadata = df_bubble.to_dict(orient="records")

print("Primul text:")
print(texts[0][:500])

Primul text:
CU TIPETE DNA JUDECATOR NU SE LUCREAZA CU SETATENILOR.REREVIZUITI COMPORTAMENTUL .😮😮😮


## 3. Generăm embeddings
Un embedding este o reprezentare vectorială a textului: texte apropiate ca sens primesc vectori apropiați în spațiul semantic.
Folosim un model multilingv, deoarece corpusul este în limba română.
Normalizăm vectorii la lungime 1, astfel încât produsul scalar din FAISS să funcționeze ca similaritate cosinus.

In [8]:
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")
print("Număr texte:", len(texts))
print("Dimensiune embeddings:", embeddings.shape)

c:\Users\User\Desktop\Analiza Datelor Complexe\AN2\curs_Inginerie_AI\proiect_AI\echochamber-project-team-4\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(messag

Număr texte: 50
Dimensiune embeddings: (50, 384)


### Verificare rapidă
Răspunde în 1–2 propoziții în notebook:
- Câte texte are bula ta?
- Câți vectori au fost generați?
- Ce înseamnă a doua valoare din `embeddings.shape`?

In [ ]:
# TODO student:
# Bula mea are 50 texte.
# Au fost generați 50 vectori.
# A doua valoare din embeddings.shape reprezintă lungimea vectorilor

## 4. Construim indexul FAISS
FAISS este biblioteca care caută rapid vectori apropiați.
Indexul nu păstrează textele originale. El păstrează doar reprezentările vectoriale.
De aceea salvăm două lucruri:
- `index.faiss` = indexul vectorial;
- `index.pkl` = textele originale și metadatele.

In [9]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
out_dir = VECTOR_DIR / slug
out_dir.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(out_dir / "index.faiss"))
with open(out_dir / "index.pkl", "wb") as f:
    pickle.dump(metadata, f)
print("Salvat în:", out_dir)
print("Vectori în index:", index.ntotal)

Salvat în: assets\vectorstores\anti_sistem
Vectori în index: 50


## 5. Verificăm fișierele create
Dacă totul a mers corect, bula ta are acum un folder propriu în `assets/vectorstores/`.
Acest folder trebuie să conțină `index.faiss` și `index.pkl`.

In [ ]:
# TODO student:
# index.faiss există: DA
# index.pkl există: DA
# index.ntotal este egal cu numărul de texte: DA

## Ce am construit?
Am transformat textele curate ale unei bule într-un index vectorial local.
Acest index nu generează răspunsuri. El doar permite căutarea semantică.
În continuare vom testa dacă, pentru o întrebare, FAISS returnează texte relevante.

## 6. Testăm retrieval-ul
Acum simulăm logica aplicației.
- Utilizatorul introduce o știre sau o afirmație politică.
- Retriever-ul caută în memoria bulei cele mai asemănătoare texte.
- Nu generăm încă un răspuns cu LLM. Doar verificăm ce exemple sunt recuperate.

In [13]:
# Text nou introdus în aplicație

input_text = "Toti din guvern sunt niste impostori, nu cred in ei."

In [14]:
# Transformăm textul nou în embedding

query_vector = model.encode(
    [input_text],
    normalize_embeddings=True
).astype("float32")

In [11]:
# query_vector

In [15]:
# Căutăm cele mai apropiate 5 texte din bula noastră

scores, results = index.search(query_vector, k=5)

for rank, pos in enumerate(results[0], start=1):
    row = metadata[pos]
    
    print(f"\nRezultat {rank}")
    print("Scor:", round(float(scores[0][rank-1]), 3))
    print("Text:", row["text"][:500])


Rezultat 1
Scor: 0.382
Text: Un ordinar precum Grindeanu, nu are ce sa caute la guvernare. Locul lui e dupa gratii, dar pare ca nu se prea vrea 😠

Rezultat 2
Scor: 0.333
Text: D-le Președinte,dacă nu aveți curajul,onoarea,puterea de a lupta cu mafia securisto-comunisto-rusofilă psdnl,dacă vă lăsați dominat de mafia securisto-rusofilă trădătoare din serviciile secrete trădătoare,aveți măcar bunul simț și plecați acasă.România nu mai poate gira incă un președinte slab,fricos,politruc,servil cu mafia securisto-rusofilă, care distruge viitorul țarii.Reacționați sau plecați!

Rezultat 3
Scor: 0.303
Text: Am impresia că Digi sunt mâhniți că regimul terorist din Iran este distrus...Se vede o stare de nemulțumire că și noi contribuim la nimicirea teroriștilor și a sponsorilor lor. Când ”Gărzile revoluționare” ucideau în masă propria populație, zeci de mii de oameni, Digi nu se scandaliza în halul ăsta, dar acum, acum e prăpădul lumii că se stinge un regim criminal. Ipocriți și influențați dem

### TODO
Schimbă `input_text` cu o afirmație potrivită pentru agentul tău.
Rulează căutarea.
Notează:
- câte rezultate din 5 sunt relevante;
- dacă textele recuperate exprimă vocea agentului;
- dacă ai observat un text slab care ar trebui eliminat.

### Cel putin 3 din cele 5 comentarii exprima vocea agentului anti-sisten